# NLP - Text Classification Lab

Note that this lab has three levels: basic, regular and advanced.


Completing the **basic** part earns you a grade of 5.5-6.0.

Completing the **regular** part earns you a max grade of 8.0.

Completing the **advanced** part earns you a max grade of 10.0.

Please return a Jupyter notebook as a submission in Canvas, to make the grading easier for us.




## Basic Level

## Necessary Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.metrics import accuracy_score
import re
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import *
from nltk.corpus import stopwords
from tqdm import tqdm

import nltk
nltk.download('stopwords')

/home/tonso/code/ptonso/course/ut/ds-ai/venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/tonso/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package stopwords to /home/tonso/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Reading the Dataset

In [ ]:
datapath = Path.cwd().parent / "databank" / "reviews.csv"
dataset=pd.read_csv(datapath)

dataset=dataset[['sentiment', 'text']]
dataset = dataset.rename(columns={'sentiment': 'class'})
dataset

,class,text
0,neg,"Now, I won't deny that when I purchased this o..."
1,neg,"The saddest thing about this ""tribute"" is that..."
2,neg,Last night I decided to watch the prequel or s...
3,neg,I have to admit that i liked the first half of...
4,neg,I was not impressed about this film especially...
...,...,...
24995,pos,"This film is fun, if your a person who likes a..."
24996,pos,After seeing this film I feel like I know just...
24997,neg,first this deserves about 5 stars due to actin...
24998,neg,If you like films that ramble with little plot...


### Exercise 1
Split the dataset into two data structures (pandas frames), one for the reviews (our documents) (X) and one for the class (y).
Check that the lengths of both dataframes are equal

In [ ]:
#Your answer goes here

X = dataset['text']
y = dataset['class']
assert len(X) == len(y), "X and y must have the same length"
print(f"Size of X is : {X.shape}, Size of y is : {y.shape}")

Size of X is : (25000,), Size of y is : (25000,)


### Preprocessing
We now begin our preprocessing task, by lowercasing all of the documents, removing special characters and numerical values. Then we tokenize and stem our documents. 
### Exercise 2
Write a function to_lower(X) that takes a dataframe and returns its content in lower case.

In [ ]:
def to_lower(X): 
    X = X.str.lower()
    return X

In [ ]:
X=to_lower(X)
X.head()

0    now, i won't deny that when i purchased this o...
1    the saddest thing about this "tribute" is that...
2    last night i decided to watch the prequel or s...
3    i have to admit that i liked the first half of...
4    i was not impressed about this film especially...
Name: text, dtype: str

### Exercise 3 
Write a function clean_text(X), that takes the dataset as an input and removes all special characters and numerical values from it.

In [ ]:
def clean_text(X):

    X = X.str.replace(r'[^a-zA-Z\s]', '', regex=True)
    X = X.str.replace(r'\s+', ' ', regex=True)
    return X

In [ ]:
X=clean_text(X)

In [ ]:
X

0        now i wont deny that when i purchased this off...
1        the saddest thing about this tribute is that a...
2        last night i decided to watch the prequel or s...
3        i have to admit that i liked the first half of...
4        i was not impressed about this film especially...
                               ...                        
24995    this film is fun if your a person who likes a ...
24996    after seeing this film i feel like i know just...
24997    first this deserves about stars due to acting ...
24998    if you like films that ramble with little plot...
24999    as interesting as a sheet of cardboard this di...
Name: text, Length: 25000, dtype: str

### Exercise 4
Write a function tokenize(X) that takes a dataframe and returns the tokens in each row (document).

In [ ]:
def tokenize(X):
    '''
    TODO: implement this function
    '''
    X = X.apply(lambda x: x.split())
    return X


In [ ]:
X=tokenize(X)
print(X)

0        [now, i, wont, deny, that, when, i, purchased,...
1        [the, saddest, thing, about, this, tribute, is...
2        [last, night, i, decided, to, watch, the, preq...
3        [i, have, to, admit, that, i, liked, the, firs...
4        [i, was, not, impressed, about, this, film, es...
                               ...                        
24995    [this, film, is, fun, if, your, a, person, who...
24996    [after, seeing, this, film, i, feel, like, i, ...
24997    [first, this, deserves, about, stars, due, to,...
24998    [if, you, like, films, that, ramble, with, lit...
24999    [as, interesting, as, a, sheet, of, cardboard,...
Name: text, Length: 25000, dtype: object

### Exercise 5
Write a function remove_stop_words(X, stop_words), that takes a dataset X and the set of stop words you want removed from it. Your function should return the dataset, free of any common words.

In [ ]:
def remove_stop_words(X, stop_words):
    stop_set = set(stop_words)
    return X.apply(lambda x: [word for word in x if word not in stop_set])


In [ ]:
X = remove_stop_words(X, stopwords.words('english'))

In [ ]:
X

0        [wont, deny, purchased, ebay, high, expectatio...
1        [saddest, thing, tribute, almost, singers, inc...
2        [last, night, decided, watch, prequel, shall, ...
3        [admit, liked, first, half, sleepers, looked, ...
4        [impressed, film, especially, fact, went, cine...
                               ...                        
24995    [film, fun, person, likes, good, campy, featur...
24996    [seeing, film, feel, like, know, little, bit, ...
24997    [first, deserves, stars, due, acting, would, g...
24998    [like, films, ramble, little, plot, exposition...
24999    [interesting, sheet, cardboard, dispensable, p...
Name: text, Length: 25000, dtype: object

### Exercise 6
Write a function stem(X), that takes a dataframe X and returns the stems of all the words in it.

You're free to choose any stemmer you want.

It's also possible to use a lemmatizer (lemmatization will be a lot slower!).

In [ ]:
def stem(X):
    stemmer = PorterStemmer()
    return X.apply(lambda x: [stemmer.stem(word) for word in x])    
    

In [ ]:
X=stem(X)

In [ ]:
X

0        [wont, deni, purchas, ebay, high, expect, incr...
1        [saddest, thing, tribut, almost, singer, inclu...
2        [last, night, decid, watch, prequel, shall, sa...
3        [admit, like, first, half, sleeper, look, good...
4        [impress, film, especi, fact, went, cinema, fa...
                               ...                        
24995    [film, fun, person, like, good, campi, featur,...
24996    [see, film, feel, like, know, littl, bit, usa,...
24997    [first, deserv, star, due, act, would, give, b...
24998    [like, film, rambl, littl, plot, exposit, spic...
24999    [interest, sheet, cardboard, dispens, period, ...
Name: text, Length: 25000, dtype: object

### Exercise 7
Having called a tokenizer and a stemmer on our dataset, the resulting rows are now of type list.
We need to convert them back to str, as our CountVectorizer expects a dataset where every document is a string. 
Define a function to_String(X), that takes your dataset and stitches back its rows back to the str format.

In [ ]:
def to_String(X): 
    X = X.apply(lambda x: ' '.join(x))
    return X

In [ ]:
X=to_String(X)
X


0        wont deni purchas ebay high expect incred outo...
1        saddest thing tribut almost singer includ othe...
2        last night decid watch prequel shall say call ...
3        admit like first half sleeper look good act ev...
4        impress film especi fact went cinema famili go...
                               ...                        
24995    film fun person like good campi featur film ev...
24996    see film feel like know littl bit usa david ly...
24997    first deserv star due act would give better su...
24998    like film rambl littl plot exposit spice kinki...
24999    interest sheet cardboard dispens period piec l...
Name: text, Length: 25000, dtype: str

### Vector Space Model
Now that we preprocessed our corpus, we can proceed to vectorize it.
### Exercise 8
We can now use the [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) from sklearn to create our document-term-matrix.

a. Create a document-term matrix from your dataset X, use min_df and max_df parameters to exclude words that appear in less than 10 documents, and words that appear in more than 99.5% of the documents. We want to keep only words of medium frequency, as stated in the lecture.

In [ ]:
vectorizer = CountVectorizer(min_df=10, max_df=0.995)
X = vectorizer.fit_transform(X)

b. Print the size and the contents of your vocab (feature space)

In [ ]:
vocab = vectorizer.get_feature_names_out()
print(f"The size of the vocabulary is: {len(vocab)}")
print("The first 10 items in the vocab are:\n", vocab[:10])

The size of the vocabulary is: 13826
The first 10 items in the vocab are:
 ['aag' 'aam' 'aaron' 'ab' 'abandon' 'abba' 'abbey' 'abbi' 'abbot' 'abbott']


### Training our Logistic Regressor

### Exercise 9:
Before we dive into training our model, let's get our vector of true labels **y** into the right format.
Notice that by printing the contents of **y** below, what we get are the labels **neg** and **pos**. 
The model works only with **1 and 0**.
Let's convert the labels accordingly.

In [ ]:
y[:10]

0    neg
1    neg
2    neg
3    neg
4    neg
5    pos
6    pos
7    neg
8    pos
9    neg
Name: class, dtype: str

a. Use the [label_ encoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html) from Sklearn, to transform the labels in the vector **y** accordingly.

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
y[:10]

array([0, 0, 0, 0, 0, 1, 1, 0, 1, 0])

We also create a test and train set from our DTM and our vector y.

In [ ]:
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.25)

### Exercise 10
Use the [Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) classifier from Sklearn to train your model, and check its accuracy on the test set

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(train_X, train_y)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [ ]:
print("Accuracy on train set", accuracy_score(train_y, model.predict(train_X)) * 100)
print("Accuracy on test set", accuracy_score(test_y, model.predict(test_X)) * 100)

Accuracy on train set 99.424
Accuracy on test set 86.176


**Our results show a clear sign of overfitting**

## Regular 

Our main goal in the regular exercises is to provide a basic implementation for the logistic regression model.


We'll first define a function **initialize(X)** to get the initial vector of weights W and bias b.

We will then do our **forward_pass(X, W, b)** to get a vector of predictions.

We then write a function **gradient_descent(X, W, b, y, lr)** to get the updated vector of weights W and bias b.


We conclude this part by writing the model function, which calls all of the functions we defined, and proceed with the learning.

### Exercise 11
Write a function **initialize(X)**, which takes a DTM as an input and returns a vector of weights W and a scalar b for the bias. Both are initialized with some random values.

In [ ]:
# train_X=train_X.toarray()
train_y=np.array(train_y)
train_y=train_y.reshape(train_y.shape[0],1)

In [ ]:
def initialize(X):
    d = X.shape[1]
    W = np.random.random(size=(d,1))
    b = np.random.random(size=1)

    return (W, b)

In [ ]:
W,b=initialize(train_X)
print("Shape of the vector W is:",W.shape)

NameError: name 'W' is not defined

### Exercise 12
Write a function **forward_pass(X, W, b)** which takes the DTM X, the vector W and the bias b as inputs and returns a vector of predictions P.

Your function should implement the following equations

$$Z=X.W + b$$
$$P=\sigma{(Z)}=\frac{1}{1+e^{-Z}}$$

You can also implement the sigmoid as a separate function, 

In [ ]:
def forward_pass(X, W, b):
    Z = X @ W + b
    return 1/(1 + np.exp(-Z))

In [ ]:
P=forward_pass(train_X, W, b)
print("The vector of predictions shape is:",P.shape)

The vector of predictions shape is: (18750, 1)


### Exercise 13
Calculating the loss/cost function
You can use the [implementation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.log_loss.html) by Sklearn, but feel free to also implement your own.
<!-- #endregion -->i

```python id="36N1AVywgUy6"
def log_loss(y_true, p_pred):
    loss = -(y_true * np.log(p_pred) + (1-y_true) * np.log(1-p_pred))
    return np.mean(loss)

```

<!-- #region id="vonvu9_UGM6x" -->
### Exercise 14
Write a function **gradient_descent(X, W, b , P, y, lr)** and returns the updated weight vector W, and bias b.
Your function needs to implement the following equations:

$$dW=\frac{1}{ne}X^T . (P-y)$$

$$db=\frac{1}{ne} \sum(P-y)$$

$$W=W-\alpha dW$$

$$b=b-\alpha db$$

In [ ]:
def gradient_descent(X, W, b, P, y, lr):
    n = X.shape[0]
    dW = 1/n * X.T @ (P - y)
    db = 1/n * np.sum(P - y)
    W = W - lr * dW
    b = b - lr * db
    return W, b

In [ ]:
W, b=gradient_descent(train_X, W, b, P, train_y, lr=0.2)
print("shape of the updated W is:", W.shape)

shape of the updated W is: (13822, 1)


### Exercise 15
Now we can implement our logistic regression model in 3 simple steps
1. initialize the vector W and the bias b

2. repeat until number of iterations is reached   
    2.1. get a vector of predictions P  
    2.2. update the weights W and bias b using gradient descent  
    
3. return the final vector W and bias b

**logistic_regression(X, y, lr, iters)**, takes the matrix X as an input, the vector of true labels y, a learning rate, and the number of iterations iters. The function returns the learned parameters of the model, namely W and b.

In [ ]:
def logistic_regression(X, y, lr, iters):
  
    W, b = initialize(X)
    

    for i in tqdm(range(iters)):
        P = forward_pass(X, W, b)
        W, b = gradient_descent(X, W, b, P, y, lr)
    
    return W, b
    

In [ ]:
W, b= logistic_regression(train_X, train_y, lr=1.5, iters=500)

100%|████████████████████████████████████████████████████████████████████████████████| 500/500 [15:37<00:00,  1.87s/it]


### Testing our model
We have been able to implement the model and run it on our training set. It's time to see how well it does. 
We'll first make a function **predict(X, W, b)**, that takes the dataset and the learned parameters and returns an array of predictions. Our threshold is 0.5, any prediction below that is returned as 0, and any above it are returned as 1.

In [ ]:
def predict(X, W, b):
    P = forward_pass(X, W, b)
    P=1*(P >= 0.5)
    return P

In [ ]:
P_train=predict(train_X, W, b)

In [ ]:
print("Accuracy on our train set is, ",accuracy_score( train_y, P_train)*100, "%")

Accuracy on our train set is,  85.58933333333333 %


In [ ]:
P_test=predict(test_X, W, b)

In [ ]:
print("Accuracy on our test set is", accuracy_score(test_y, P_test)*100, "%")

Accuracy on our test set is 80.88 %


## Advanced

In this part, we set to understand what did the model actually learn.

### Exercise 16
Using the CountVectorizer of Sklearn, recreate a pandas frame where the rows contain the documents and the columns contain the features. 



In [ ]:
#Your code goes here
DTM = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())


In [ ]:
DTM

,aag,aam,aaron,ab,abandon,abba,abbey,abbi,abbot,abbott,...,zombi,zombiesbr,zone,zoo,zoom,zorro,zu,zucker,zulu,zuniga
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Exercise 17
Knowing that our logistic regressor learns a weight for each feature (word in the vocab),  return the words with the highest weights (5 highest), and the words with lowest weights (5 lowest).

In [ ]:

weights = model.coef_[0]
vocab = vectorizer.get_feature_names_out()
weights_df = pd.DataFrame({'word': vocab, 'weight': weights})
sorted_weights = weights_df.sort_values(by='weight')

print('5 words with the lowest weights')
for index, row in sorted_weights.head(5).iterrows():
    print(f"the word `{row['word']}` has weight {row['weight']:.2f}")
    
print('__________________')
print('5 words with the highest weights')
for index, row in sorted_weights.tail(5).iterrows():
    print(f"the word `{row['word']}` has weight {row['weight']:.2f}")


### Exercise 18
Print the weights of the words "good" and "bad"

In [ ]:
#Your code goes here
bad_weight = weights_df.loc[weights_df['word'] == 'bad', 'weight'].values[0]
good_weight = weights_df.loc[weights_df['word'] == 'good', 'weight'].values[0]

print(f"weight of word bad [{bad_weight}]")
print(f"weight of word good [{good_weight}]")


##### Hope you enjoyed learning about logistic regression!